# v2 — 03: Pipeline & Cleaning
**Iteration 2** extends the v1 pipeline with:
- Draft capital features (pick, round, age at draft)
- Combine athleticism (40-yard dash, weight, vertical)
- Injury history (games missed, IR flag, career durability rate)
- College production (final season rec yards, TDs, dominator rate)
- **Rookies included** with NFL stat features set to 0

**Prerequisites:** Run `v1_01_data_gathering.ipynb` and `v2_01_data_gathering.ipynb` first.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from src.fetchers.nfl_fetcher import (
    fetch_seasonal_stats, fetch_rosters, fetch_snap_counts,
    fetch_draft_picks, fetch_combine_data, fetch_injuries,
)
from src.fetchers.sleeper_fetcher import fetch_players
from src.fetchers.college_fetcher import fetch_college_stats
from src.pipeline.cleaner import build_clean_dataset, read_from_db
from src.pipeline.features import build_model_dataset

pd.set_option('display.max_columns', 40)

## 1. Load All Data Sources (from cache)

In [ ]:
# Set CFBD API key before fetching college stats
# Get your key at https://collegefootballdata.com/key
import os
os.environ["CFBD_API_KEY"] = "your_key_here"  # <-- replace with your actual key

In [2]:
# v1 sources
seasonal     = fetch_seasonal_stats()
rosters      = fetch_rosters()
snap_counts  = fetch_snap_counts()
sleeper      = fetch_players()

# v2 sources
draft_picks  = fetch_draft_picks()
combine_data = fetch_combine_data()
injuries     = fetch_injuries()
college_stats = fetch_college_stats(draft_seasons=list(range(2016, 2025)))

print(f'Seasonal:     {seasonal.shape}')
print(f'Rosters:      {rosters.shape}')
print(f'Snap counts:  {snap_counts.shape}')
print(f'Draft picks:  {draft_picks.shape}')
print(f'Combine:      {combine_data.shape}')
print(f'Injuries:     {injuries.shape}')
print(f'College:      {college_stats.shape}')

  [1/13] 2012 — loaded from checkpoint
  [2/13] 2013 — loaded from checkpoint
  [3/13] 2014 — loaded from checkpoint
  [4/13] 2015 — loaded from checkpoint
  [5/13] 2016 — loaded from checkpoint
  [6/13] 2017 — loaded from checkpoint
  [7/13] 2018 — loaded from checkpoint
  [8/13] 2019 — loaded from checkpoint
  [9/13] 2020 — loaded from checkpoint
  [10/13] 2021 — loaded from checkpoint
  [11/13] 2022 — loaded from checkpoint
  [12/13] 2023 — loaded from checkpoint
  [13/13] 2024 — loaded from checkpoint
Combined seasonal stats: 7870 rows saved to /home/ruppdj/claude_testing/fantasy_football/src/fetchers/../../data/raw/seasonal_stats.parquet
Loading rosters from cache: /home/ruppdj/claude_testing/fantasy_football/src/fetchers/../../data/raw/rosters.parquet
  [1/13] 2012 — loaded from checkpoint
  [2/13] 2013 — loaded from checkpoint
  [3/13] 2014 — loaded from checkpoint
  [4/13] 2015 — loaded from checkpoint
  [5/13] 2016 — loaded from checkpoint
  [6/13] 2017 — loaded from checkpoin

## 2. Run v2 Cleaning Pipeline

In [3]:
clean_df = build_clean_dataset(
    seasonal_df=seasonal,
    rosters_df=rosters,
    sleeper_df=sleeper,
    snap_df=snap_counts,
    draft_df=draft_picks,
    combine_df=combine_data,
    injury_df=injuries,
    college_df=college_stats,
)
print(f'\nClean dataset: {clean_df.shape}')
clean_df.head(3)

Cleaning seasonal stats...
Merging roster metadata...
Merging Sleeper metadata...
Merging snap pct...
Computing age at season...
Merging draft data...
Merging combine data...
Merging injury data...
Merging college stats...
Loading to SQLite (nfl_stats)...
Loaded 5711 rows into table 'nfl_stats'
Clean dataset: 5711 rows, 85 columns

Clean dataset: (5711, 85)


,player_id,season,season_type,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,...,pfr_id,status,injury_status,depth_chart_order,search_rank,avg_snap_pct,age_at_season,draft_pick,draft_round,age_at_draft,cfb_player_id,draft_season,is_undrafted,combine_forty,combine_weight,combine_height,combine_vertical,combine_bench,games_missed,ir_flag
0,00-0004541,2012,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.00000,0,0.000000,0.000000,0,...,DrivDo00,NaN,NaN,NaN,NaN,NaN,37.6,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,0.0,0.0
1,00-0006101,2012,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.00000,0,0.000000,0.000000,0,...,GonzTo00,NaN,NaN,NaN,NaN,NaN,36.5,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,0.0,0.0
2,00-0007091,2012,REG,138,221,1367.0,7,5.0,14.0,103.0,2,1,1707.0,557.0,71.0,-14.85631,1,6.316032,0.371353,13,...,HassMa00,Inactive,None,NaN,9999999.0,NaN,36.9,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Inspect New Feature Coverage

In [4]:
new_v2_cols = [
    'draft_pick', 'draft_round', 'age_at_draft', 'is_undrafted',
    'combine_forty', 'combine_weight', 'combine_vertical',
    'games_missed', 'ir_flag',
    'college_rec_yards', 'college_dominator_rate',
]
print('Coverage of new v2 features:')
for col in new_v2_cols:
    if col in clean_df.columns:
        pct = clean_df[col].notna().mean()
        print(f'  {col:35s}: {pct:.1%} non-null')
    else:
        print(f'  {col:35s}: NOT IN DATASET')

Coverage of new v2 features:
  draft_pick                         : 53.9% non-null
  draft_round                        : 53.9% non-null
  age_at_draft                       : 53.9% non-null
  is_undrafted                       : 100.0% non-null
  combine_forty                      : 50.1% non-null
  combine_weight                     : 55.1% non-null
  combine_vertical                   : 47.0% non-null
  games_missed                       : 76.7% non-null
  ir_flag                            : 76.7% non-null
  college_rec_yards                  : NOT IN DATASET
  college_dominator_rate             : NOT IN DATASET


## 4. Feature Engineering (with Rookie Inclusion)

In [5]:
model_df = build_model_dataset(clean_df)
print(f'\nModel-ready dataset: {model_df.shape}')
model_df.head(3)

Adding rookie flag...
Adding lagged PPR features (rookies → 0)...
Adding lagged injury features...
Adding target variable...
Adding usage features...
Encoding position...
Selecting model features...
Model-ready dataset: 4058 rows, 37 features (1211 rookies included)
Saving model_ready table to SQLite...
Saved 4058 rows to model_ready table

Model-ready dataset: (4058, 37)


,player_id,full_name,position,season,ppr_pts_next,is_rookie,age_at_season,ppr_pts_prev1,ppr_pts_prev2,fantasy_points_ppr,ppr_per_game,td_rate,target_share,air_yards_share,targets_per_game,carries_per_game,avg_snap_pct,receiving_epa,rushing_epa,passing_epa,racr,games_missed_prev1,career_games_missed_rate,ir_flag_prev1,draft_pick,draft_round,age_at_draft,is_undrafted,combine_forty,combine_weight,combine_height,combine_vertical,combine_bench,pos_QB,pos_RB,pos_WR,pos_TE
1,00-0006101,Tony Gonzalez,TE,2012,218.90,1,36.5,0.00,0.0,234.00,14.625,0.500,3.171097,3.236721,7.75,0.000,NaN,61.123664,0.000000,0.000000,16.35189,0.0,0.0,0.0,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,0,0,0,1
2,00-0007091,Matt Hasselbeck,QB,2012,16.94,1,36.9,0.00,0.0,76.48,9.560,0.875,0.000000,0.000000,0.00,1.625,NaN,0.000000,-3.944240,-14.856310,0.00000,0.0,0.0,0.0,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,1,0,0,0
847,00-0007091,Matt Hasselbeck,QB,2014,91.10,0,38.9,76.48,0.0,16.94,4.235,0.500,0.000000,0.000000,0.00,2.000,0.305,0.000000,-7.756981,-0.628051,0.00000,0.0,0.0,0.0,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,1,0,0,0


## 5. Validate Rookie Inclusion

In [6]:
if 'is_rookie' in model_df.columns:
    rookies = model_df[model_df['is_rookie'] == 1]
    vets = model_df[model_df['is_rookie'] == 0]
    print(f'Rookies in model dataset: {len(rookies)}')
    print(f'Veterans in model dataset: {len(vets)}')
    print(f'\nRookie ppr_pts_prev1 (should all be 0): {rookies["ppr_pts_prev1"].unique()}')
    print(f'\nSample rookies:')
    cols = ['full_name', 'position', 'season', 'draft_pick', 'age_at_draft',
            'college_rec_yards', 'ppr_pts_next']
    print(rookies[[c for c in cols if c in rookies.columns]].head(10).to_string(index=False))

Rookies in model dataset: 1211
Veterans in model dataset: 2847

Rookie ppr_pts_prev1 (should all be 0): [0.]

Sample rookies:
      full_name position  season  draft_pick  age_at_draft  ppr_pts_next
  Tony Gonzalez       TE    2012         NaN           NaN        218.90
Matt Hasselbeck       QB    2012         NaN           NaN         16.94
 Peyton Manning       QB    2012         NaN           NaN        409.98
Brandon Stokley       WR    2012         NaN           NaN         24.50
      Tom Brady       QB    2012         NaN           NaN        251.52
   Michael Vick       QB    2012         NaN           NaN        101.20
    Steve Smith       WR    2012         NaN           NaN        162.50
   Santana Moss       WR    2012         NaN           NaN         99.70
   Reggie Wayne       WR    2012         NaN           NaN        102.80
     Drew Brees       QB    2012         NaN           NaN        357.68


## 6. Null Rates in Model Features

In [7]:
null_rates = model_df.isna().mean().sort_values(ascending=False)
print('Feature null rates (top 20):')
print(null_rates[null_rates > 0].head(20))

Feature null rates (top 20):
combine_bench       0.634549
combine_vertical    0.516511
combine_forty       0.480286
draft_round         0.443568
draft_pick          0.443568
age_at_draft        0.443568
combine_height      0.435683
combine_weight      0.434943
avg_snap_pct        0.080089
dtype: float64
